# EC581 Trend Following — One-Button Runner

**How to use:** from the menu pick **Run → Run All Cells** (or **Kernel → Restart Kernel and Run All Cells**). That is the only thing you need to do.

This notebook rebuilds every market-data file from the bundled **`data.csv`** (fully **offline** — no internet / Yahoo Finance needed) and then runs the **entire project, every step, for both strategies — HP first, then LOWESS** — printing every result table right below each cell.

**Pipeline order**

1. `data` — rebuild the BIST100 dataset from `data.csv`
2. `sweep` — Phase-1 grid search (S1–S4 on the BIST100 index; covers both strategies at once)
3. **HP:** `phase2` → `phase2-mc` → `phase2-portfolio` → `walkforward`
4. **LOWESS:** `phase2` → `phase2-mc` → `phase2-portfolio` → `walkforward`

> **Kernel:** make sure the kernel (top-right) is the project's **`.venv`**. The heavy computation always runs through `.venv/bin/python`, but this notebook needs `pandas` available to load `data.csv` and display tables.


In [ ]:
# ============================================================
#  ONE SETTING — that is all you can change.
# ============================================================
SMOKE = False   # True  = quick ~couple-minutes test on just 5 stocks (try this first).
                # False = full BIST100 universe — the real project run.
                #         WARNING: the full run is heavy. The LOWESS walk-forward
                #         step in particular can take a long time (tens of minutes
                #         to hours). Leave SMOKE = True for a fast end-to-end check.
# ============================================================

import os, sys, subprocess
from pathlib import Path

# Find the repo root (the folder that holds config.py and data.csv), no matter
# where the notebook was launched from.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "config.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

try:
    import pandas as pd
except ImportError:
    raise SystemExit(
        "pandas is not available in this kernel. Use the kernel picker (top-right) "
        "to select the project's .venv environment, then Run All again."
    )

import config
try:
    from IPython.display import display
except Exception:
    display = print

DATA_CSV     = REPO_ROOT / "data.csv"
RESULTS_DIR  = config.get_dataset("bist100").results_dir
VENV_PY      = REPO_ROOT / ".venv" / "bin" / "python"
PYTHON       = str(VENV_PY) if VENV_PY.exists() else sys.executable

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Each friendly step name -> the module main.py would forward to.
MODULES = {
    "data":             "src.data.build",
    "sweep":            "src.backtest.run_sweep",
    "phase2":           "src.eval.run_phase2",
    "phase2-mc":        "src.eval.run_phase2_mc",
    "phase2-portfolio": "src.eval.run_phase2_portfolio",
    "walkforward":      "src.eval.run_walkforward",
}


def restore_raw_from_csv(force=False):
    """Recreate data/raw/*.parquet from data.csv - the offline stand-in for
    fetching Yahoo Finance. Idempotent: skips tickers already present."""
    if not DATA_CSV.exists():
        raise FileNotFoundError(
            f"{DATA_CSV} not found. It must sit next to this notebook; it holds "
            "all the market data the pipeline needs."
        )
    print(f"[run] loading market data from {DATA_CSV.name} (offline) ...")
    df = pd.read_csv(DATA_CSV, parse_dates=["date"])
    written = skipped = 0
    for ticker, g in df.groupby("ticker", sort=True):
        out = config.RAW_DIR / f"{ticker}.parquet"
        if out.exists() and not force:
            skipped += 1
            continue
        frame = (g.drop(columns=["ticker"]).set_index("date")
                  .sort_index()[config.OHLCV_COLS])
        frame.index.name = "date"
        frame.to_parquet(out)
        written += 1
    print(f"[run] raw data ready: wrote {written}, kept {skipped} existing "
          f"({df['ticker'].nunique()} tickers).")


def _mtimes():
    return {p: p.stat().st_mtime for p in RESULTS_DIR.glob("*.parquet")}


def _show_new(before):
    """Display every results parquet this step just created or updated."""
    after = _mtimes()
    fresh = sorted(p for p, m in after.items()
                   if p not in before or m > before[p] + 1e-6)
    for p in fresh:
        d = pd.read_parquet(p)
        print(f"\nRESULT  {p.relative_to(REPO_ROOT)}   shape={d.shape}")
        display(d.head(40))
        if len(d) > 40:
            print(f"... {len(d) - 40} more rows in {p.name}")


def _argv(command, strategy):
    """Build the exact argument list main.py would forward to the module."""
    argv = ["--dataset", "bist100"]
    if SMOKE and command != "sweep":
        argv.append("--smoke")
    if command == "data":
        argv += ["--source", "yfinance"]            # no-op: raw/ already restored
    elif command == "sweep":
        argv += ["--min-trades", "30"]
    elif command == "walkforward":
        argv += ["--strategy", strategy, "--min-trades", "30"]
    else:                                            # phase2 / -mc / -portfolio
        argv += ["--strategy", strategy]
        if command == "phase2-mc":
            argv += ["--n-iter", "1000"]
    return argv


def run_step(command, strategy=None):
    """Run one pipeline step through the venv interpreter, stream its output
    into this cell, then display the result tables it produced."""
    argv  = _argv(command, strategy)
    label = command + (f"  [{strategy}]" if strategy else "")
    cli_eq = "python main.py " + command + " " + " ".join(
        a for a in argv if a not in ("--source", "yfinance"))
    print("=" * 74)
    print(f"STEP: {label}   (smoke={SMOKE})")
    print("CLI equivalent:  " + cli_eq.strip())
    print("=" * 74)
    before = _mtimes()
    cmd = [PYTHON, "-u", "-m", MODULES[command], *argv]
    proc = subprocess.Popen(cmd, cwd=REPO_ROOT, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        print(f"[run] !! step {label!r} exited with code {rc}")
    else:
        _show_new(before)
    return rc


print("Mode:", "SMOKE - fast 5-stock test" if SMOKE
      else "FULL BIST100 universe - the real run (can be slow; LOWESS walk-forward is the slowest step)")
restore_raw_from_csv()


## Step 1 — Build the dataset from `data.csv` (offline)

In [ ]:
run_step("data")

## Step 2 — Phase-1 sweep (S1–S4 on the BIST100 index)

Strategy-independent: this single sweep scores all four strategies at once.

In [ ]:
run_step("sweep")

## Strategy 1 of 2 — HP (Harris–Yilmaz low-frequency filter)

In [ ]:
run_step("phase2", "hp")

In [ ]:
run_step("phase2-mc", "hp")

In [ ]:
run_step("phase2-portfolio", "hp")

In [ ]:
run_step("walkforward", "hp")

## Strategy 2 of 2 — LOWESS

> The LOWESS walk-forward (last cell) is the slowest step in the whole project on the full universe — be patient.

In [ ]:
run_step("phase2", "lowess")

In [ ]:
run_step("phase2-mc", "lowess")

In [ ]:
run_step("phase2-portfolio", "lowess")

In [ ]:
run_step("walkforward", "lowess")

---
**Done.** Every step ran for both strategies; result tables are shown above and saved under `results/` (LOWESS variants are the `*_lowess_*` files).